# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata information
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List available record sets and their IDs
record_sets_info = []
for rs in dataset.record_sets:
    print(f"RecordSet Name: {rs.name}, @id: {rs.id}")
    record_sets_info.append({'name': rs.name, 'id': rs.id})
    # List available fields/columns
    print("Fields/Columns:")
    for fld in rs.fields:
        print(f" - {fld.name} (@id: {fld.id}) [type: {fld.data_type}]")
    print()


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['id'] for rs in record_sets_info]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"RecordSet {rs_id} DataFrame columns:")
    print(df.columns.tolist())
    print(df.head(), end="\n\n")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field from the main survey record set and perform basic EDA: filtering, normalization, and grouping.

In [ ]:
# Identify main record set (most columns & survey responses)
main_rs = None
max_cols = 0
for rs_id, df in dataframes.items():
    if len(df.columns) > max_cols:
        max_cols = len(df.columns)
        main_rs = rs_id

df = dataframes[main_rs]

# Find numeric fields (likely assessment scores such as GAD-7, PHQ-9, PSQ)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    # Try to infer numeric fields (those with digits in the name?)
    numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['gad', 'phq', 'psq'])]

print("Numeric fields in main record set:", numeric_fields)

# Use GAD-7 as example numeric field (@id referencing)
selected_numeric_field = numeric_fields[0] if numeric_fields else df.columns[0]

threshold = 10
filtered_df = df[df[selected_numeric_field] > threshold]
print(f"Filtered records with {selected_numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
normalized_col = f"{selected_numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[selected_numeric_field] - filtered_df[selected_numeric_field].mean()) / filtered_df[selected_numeric_field].std()
print(f"Normalized {selected_numeric_field} for filtered records:")
print(filtered_df[[selected_numeric_field, normalized_col]].head())

# Find a categorical field for grouping (e.g., gender, education)
group_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower()]
group_field = group_fields[0] if group_fields else df.columns[0]
print(f"Grouping field chosen: {group_field}")
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[selected_numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {selected_numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions for a numeric score, and relationships between a numeric field and a demographic group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric score
plt.figure(figsize=(8,4))
sns.histplot(df[selected_numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {selected_numeric_field}")
plt.xlabel(selected_numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot for numeric field by group_field
if group_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field], y=df[selected_numeric_field])
    plt.title(f"{selected_numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(selected_numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² Mental Health Survey dataset, explored its structure by record sets and fields, extracted data for analysis, and performed preliminary EDA including filtering and normalization.

We observed distributions (e.g., GAD-7 scores) and relationships (e.g., scores by gender or education), providing insight into mental health patterns within Kilifi County. This workflow can be extended to more complex analyses and modeling tasks as needed.